In [218]:
import pandas as pd
import numpy as np 
import requests
import zipfile
import io
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [219]:
data = pd.read_csv("./bank+marketing/bank/bank-full.csv", sep=';')
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [220]:
data.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')

In [221]:
# age,
# job,
# marital,
# education,
# balance,
# housing,
# contact,
# day,
# month,
# duration,
# campaign,
# pdays,
# previous,
# poutcome,
# y

# to exclude 'default', 'loan'

In [222]:
df = data.drop(columns=['default', 'loan']).copy()
df.shape

(45211, 15)

In [223]:
df.head()

,age,job,marital,education,balance,housing,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,2143,yes,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,29,yes,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,2,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,1506,yes,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,1,no,unknown,5,may,198,1,-1,0,unknown,no


In [224]:
df.y.value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [225]:
df.dtypes

age           int64
job          object
marital      object
education    object
balance       int64
housing      object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

In [226]:
df.columns

Index(['age', 'job', 'marital', 'education', 'balance', 'housing', 'contact',
       'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome',
       'y'],
      dtype='object')

In [227]:
categorical = ['job', 'marital', 'education',  'housing', 'contact', 'month', 'poutcome']
numerical = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

   Input variables:
   # bank client data:
  - 1 - age (numeric)
  - 2 - job : type of job (categorical: "admin.","unknown","unemployed","management","housemaid","entrepreneur","student",
                                       "blue-collar","self-employed","retired","technician","services") 
  - 3 - marital : marital status (categorical: "married","divorced","single"; note: "divorced" means divorced or widowed)
  - 4 - education (categorical: "unknown","secondary","primary","tertiary")
  - 5 - default: has credit in default? (binary: "yes","no")
  - 6 - balance: average yearly balance, in euros (numeric) 
  - 7 - housing: has housing loan? (binary: "yes","no")
  - 8 - loan: has personal loan? (binary: "yes","no")
   # related with the last contact of the current campaign:
  - 9 - contact: contact communication type (categorical: "unknown","telephone","cellular") 
  - 10 - day: last contact day of the month (numeric)
  - 11 - month: last contact month of year (categorical: "jan", "feb", "mar", ..., "nov", "dec")
  - 12 - duration: last contact duration, in seconds (numeric)
   # other attributes:
  - 13 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
  - 14 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)
  - 15 - previous: number of contacts performed before this campaign and for this client (numeric)
  - 16 - poutcome: outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")

  Output variable (desired target):
  - 17 - y - has the client subscribed a term deposit? (binary: "yes","no")

In [228]:
df.isnull().any().sum()

0

## Question 1

In [229]:
df.education.mode()[0]

'secondary'

In [230]:
df.education.value_counts()

education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

In [231]:
unique_features = list(df.education.unique())
unique_features

['tertiary', 'secondary', 'unknown', 'primary']

In [232]:
for f in unique_features:
    print(f, df[df['education'] == f].count()[0])

tertiary 13301
secondary 23202
unknown 1857
primary 6851


C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_11816\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f, df[df['education'] == f].count()[0])
C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_11816\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f, df[df['education'] == f].count()[0])
C:\Users\MUHAMMED IBRAHIM\AppData\Local\Temp\ipykernel_11816\1239316148.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by posi

## Question 2

In [233]:
df['age'].corr(df['balance'])

0.09778273937134742

In [234]:
df['day'].corr(df['campaign'])

0.1624902163261929

In [235]:
df['day'].corr(df['pdays'])

-0.09304407377294044

In [236]:
# the biggest correlation: pdays and previous
df['pdays'].corr(df['previous'])

0.4548196354805016

### Target encoding

In [237]:
(df['y'] == 'yes').astype(int).value_counts()

y
0    39922
1     5289
Name: count, dtype: int64

In [238]:
df['y'] = (df['y'] == 'yes').astype(int)

## Split of the data

In [239]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [240]:
len(df_train), len(df_val), len(df_test)

(27126, 9042, 9043)

In [241]:
df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.y.values
y_val = df_val.y.values
y_test = df_test.y.values

# del df_full_train['y']
# del df_train['y']
# del df_val['y']
# del df_test['y']

## Question 3

In [242]:
from sklearn.metrics import mutual_info_score

def mutual_info_subscribtion_score(series):
    return round(mutual_info_score(series, df_train.y), 2)
 
mi = df_train[categorical].apply(mutual_info_subscribtion_score)
mi

job          0.01
marital      0.00
education    0.00
housing      0.01
contact      0.01
month        0.03
poutcome     0.03
dtype: float64

In [243]:
mi.sort_values(ascending=False)

month        0.03
poutcome     0.03
job          0.01
housing      0.01
contact      0.01
marital      0.00
education    0.00
dtype: float64

## Question 4

In [244]:
del df_full_train['y']
del df_train['y']
del df_val['y']
del df_test['y']

In [245]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')
val_dict = df_val[categorical + numerical].to_dict(orient='records')
train_dict[0]

{'job': 'technician',
 'marital': 'single',
 'education': 'tertiary',
 'housing': 'yes',
 'contact': 'cellular',
 'month': 'aug',
 'poutcome': 'unknown',
 'age': 32,
 'balance': 1100,
 'day': 11,
 'duration': 67,
 'campaign': 1,
 'pdays': -1,
 'previous': 0}

In [246]:
from sklearn.feature_extraction import DictVectorizer
dv_v = DictVectorizer(sparse=False)

dv_v.fit(train_dict)

dv_v.get_feature_names_out()

array(['age', 'balance', 'campaign', 'contact=cellular',
       'contact=telephone', 'contact=unknown', 'day', 'duration',
       'education=primary', 'education=secondary', 'education=tertiary',
       'education=unknown', 'housing=no', 'housing=yes', 'job=admin.',
       'job=blue-collar', 'job=entrepreneur', 'job=housemaid',
       'job=management', 'job=retired', 'job=self-employed',
       'job=services', 'job=student', 'job=technician', 'job=unemployed',
       'job=unknown', 'marital=divorced', 'marital=married',
       'marital=single', 'month=apr', 'month=aug', 'month=dec',
       'month=feb', 'month=jan', 'month=jul', 'month=jun', 'month=mar',
       'month=may', 'month=nov', 'month=oct', 'month=sep', 'pdays',
       'poutcome=failure', 'poutcome=other', 'poutcome=success',
       'poutcome=unknown', 'previous'], dtype=object)

In [247]:
X_train = dv_v.transform(train_dict)
X_val = dv_v.transform(val_dict)
X_val.shape

(9042, 47)

In [248]:
from sklearn.linear_model import LogisticRegression
 
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [249]:
training_result_prob = model.predict_proba(X_train)

In [250]:
training_result = model.predict(X_train)

In [251]:
training_result_prob

array([[0.96836658, 0.03163342],
       [0.96723941, 0.03276059],
       [0.93624865, 0.06375135],
       ...,
       [0.99046284, 0.00953716],
       [0.24530671, 0.75469329],
       [0.86362667, 0.13637333]])

In [252]:
training_result

array([0, 0, 0, ..., 0, 1, 0])

In [253]:
y_train

array([0, 0, 0, ..., 0, 1, 0])

In [254]:
round((training_result == y_train).mean(), 2)

0.9

In [255]:
y_pred = model.predict(X_val)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [256]:
y_pred_prob = model.predict_proba(X_val)[:, 1]
y_pred_prob

array([0.01220085, 0.0110267 , 0.14325167, ..., 0.05389903, 0.01008253,
       0.28511635])

In [257]:
base_acc = ((y_pred_prob >= 0.5) == y_val).mean()
base_acc

0.900464499004645

In [258]:
y_val

array([0, 0, 1, ..., 0, 0, 1])

In [259]:
round((y_pred == y_val).mean(), 2)

0.9

In [260]:
y_pred.shape

(9042,)

In [261]:
X_val.shape

(9042, 47)

## Question 5

In [262]:
to_remove_list = ['age', 'balance', 'previous']

numerical_variants = {}

for feat in to_remove_list:
    numerical_variants[f"without_{feat}"] = [x for x in numerical if x != feat]

# Access examples
print(numerical_variants["without_age"])
print(numerical_variants["without_balance"])
print(numerical_variants["without_previous"])

['balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
['age', 'day', 'duration', 'campaign', 'pdays', 'previous']
['age', 'balance', 'day', 'duration', 'campaign', 'pdays']


In [263]:
categorical_without_marital = ['job', 'education', 'housing', 'contact', 'month', 'poutcome']
categorical_without_marital

['job', 'education', 'housing', 'contact', 'month', 'poutcome']

In [264]:
train_dict_without_age = df_train[categorical + numerical_variants["without_age"]].to_dict(orient='records')
val_dict_without_age = df_val[categorical + numerical_variants["without_age"]].to_dict(orient='records')

train_dict_without_balance = df_train[categorical + numerical_variants["without_balance"]].to_dict(orient='records')
val_dict_without_balance = df_val[categorical + numerical_variants["without_balance"]].to_dict(orient='records')

train_dict_without_previous = df_train[categorical + numerical_variants["without_previous"]].to_dict(orient='records')
val_dict_without_previous = df_val[categorical + numerical_variants["without_previous"]].to_dict(orient='records')

train_dict_without_marital = df_train[categorical_without_marital + numerical].to_dict(orient='records')
val_dict_without_marital = df_val[categorical_without_marital + numerical].to_dict(orient='records')

In [265]:
train_dict_without_previous[0]

{'job': 'technician',
 'marital': 'single',
 'education': 'tertiary',
 'housing': 'yes',
 'contact': 'cellular',
 'month': 'aug',
 'poutcome': 'unknown',
 'age': 32,
 'balance': 1100,
 'day': 11,
 'duration': 67,
 'campaign': 1,
 'pdays': -1}

In [266]:
from sklearn.feature_extraction import DictVectorizer

dv_without_age = DictVectorizer(sparse=False)
dv_without_age.fit(train_dict_without_age)

dv_without_balance = DictVectorizer(sparse=False)
dv_without_balance.fit(train_dict_without_balance)

dv_without_previous = DictVectorizer(sparse=False)
dv_without_previous.fit(train_dict_without_previous)

dv_without_marital = DictVectorizer(sparse=False)
dv_without_marital.fit(train_dict_without_marital)

DictVectorizer(sparse=False)

In [267]:
X_train_without_age = dv_without_age.transform(train_dict_without_age)
X_val_without_age = dv_without_age.transform(val_dict_without_age)

X_train_without_balance = dv_without_balance.transform(train_dict_without_balance)
X_val_without_balance = dv_without_balance.transform(val_dict_without_balance)

X_train_without_previous = dv_without_previous.transform(train_dict_without_previous)
X_val_without_previous = dv_without_previous.transform(val_dict_without_previous)

X_train_without_marital = dv_without_marital.transform(train_dict_without_marital)
X_val_without_marital = dv_without_marital.transform(val_dict_without_marital)

In [268]:
from sklearn.linear_model import LogisticRegression
 
model_without_age = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_without_age.fit(X_train_without_age, y_train)

model_without_balance = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_without_balance.fit(X_train_without_balance, y_train)

model_without_previous = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_without_previous.fit(X_train_without_previous, y_train)

model_without_marital = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_without_marital.fit(X_train_without_marital, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [269]:
y_pred_without_age = model_without_age.predict_proba(X_val_without_age)[:, 1]

y_pred_without_balance = model_without_balance.predict_proba(X_val_without_balance)[:, 1]

y_pred_without_previous = model_without_previous.predict_proba(X_val_without_previous)[:, 1]

y_pred_without_marital = model_without_marital.predict_proba(X_val_without_marital)[:, 1]

In [270]:
base_acc - ((y_pred_without_age >= 0.5) == y_val).mean()

0.0001105950011059953

In [271]:
base_acc - ((y_pred_without_balance >= 0.5) == y_val).mean()

-0.00033178500331787486

In [272]:
base_acc - ((y_pred_without_previous >= 0.5) == y_val).mean()

-0.00044238000442375913

In [273]:
base_acc - ((y_pred_without_marital >= 0.5) == y_val).mean()

-0.0006635700066357497

In [274]:
def get_X_train_and_X_val(features):
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    return X_train, X_val

def train_model(X_train):
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return model

def validation_accuracy(X_val):
    y_pred = model.predict_proba(X_val)[:, 1]
    return ((y_pred >= 0.5) == y_val).mean().item()

In [275]:
X_train, X_val = get_X_train_and_X_val(categorical + numerical)
model = train_model(X_train)
all_features_y_acc = validation_accuracy(X_val)
all_features_y_acc

0.900464499004645

In [276]:
base_acc

0.900464499004645

In [277]:
for eliminated_feature in ["age", "balance", "marital", "previous"]:
    features = (categorical + numerical)
    features.remove(eliminated_feature)
    X_train, X_val = get_X_train_and_X_val(features)
    model = train_model(X_train)
    acc = validation_accuracy(X_val)
    print(eliminated_feature, ": ", abs(acc - all_features_y_acc))

age :  0.0001105950011059953
balance :  0.00033178500331787486
marital :  0.0006635700066357497
previous :  0.00044238000442375913


## Question 6

In [281]:
C_values = [0.01, 0.1, 1, 10, 100]


for C in C_values:

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred_prob = model.predict_proba(X_val)[:, 1]
    res = ((y_pred_prob >= 0.5) == y_val).mean().item()
    print(C, ": ", res)

0.01 :  0.8985843839858438
0.1 :  0.9006856890068569
1 :  0.9009068790090687
10 :  0.9011280690112807
100 :  0.9011280690112807
